#  Learning Unsupervised Embeddings for Molecules

In this tutorial, we will use a `SeqToSeq` model to generate fingerprints for classifying molecules.  This is based on the following paper, although some of the implementation details are different: Xu et al., "Seq2seq Fingerprint: An Unsupervised Deep Molecular Embedding for Drug Discovery" (https://doi.org/10.1145/3107411.3107424).

## Colab

This tutorial and the rest in this sequence can be done in Google colab. If you'd like to open this notebook in colab, you can use the following link.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepchem/deepchem/blob/master/examples/tutorials/Learning_Unsupervised_Embeddings_for_Molecules.ipynb)



In [13]:
# !pip install --pre deepchem
import deepchem
deepchem.__version__

'2.8.1.dev'

In [ ]:
import numpy as np
import time

import deepchem as dc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ExponentialLR, OneCycleLR
from torch.optim import Adam, AdamW
from torch.cuda.amp import GradScaler, autocast

from deepchem.models.optimizers import Adam as dc_Adam
from deepchem.models.optimizers import ExponentialDecay as dc_ExponentialDecay

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named '

In [2]:
torch.manual_seed(0)
np.random.seed(0)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# Learning Embeddings with SeqToSeq

Many types of models require their inputs to have a fixed shape.  Since molecules can vary widely in the numbers of atoms and bonds they contain, this makes it hard to apply those models to them.  We need a way of generating a fixed length "fingerprint" for each molecule.  Various ways of doing this have been designed, such as the Extended-Connectivity Fingerprints (ECFPs) we used in earlier tutorials.  But in this example, instead of designing a fingerprint by hand, we will let a `SeqToSeq` model learn its own method of creating fingerprints.

A `SeqToSeq` model performs sequence to sequence translation.  For example, they are often used to translate text from one language to another.  It consists of two parts called the "encoder" and "decoder".  The encoder is a stack of recurrent layers.  The input sequence is fed into it, one token at a time, and it generates a fixed length vector called the "embedding vector".  The decoder is another stack of recurrent layers that performs the inverse operation: it takes the embedding vector as input, and generates the output sequence.  By training it on appropriately chosen input/output pairs, you can create a model that performs many sorts of transformations.

In this case, we will use SMILES strings describing molecules as the input sequences.  We will train the model as an autoencoder, so it tries to make the output sequences identical to the input sequences.  For that to work, the encoder must create embedding vectors that contain all information from the original sequence.  That's exactly what we want in a fingerprint, so perhaps those embedding vectors will then be useful as a way to represent molecules in other models!

Let's start by loading the data.  We will use the MUV dataset.  It includes 74,501 molecules in the training set, and 9313 molecules in the validation set, so it gives us plenty of SMILES strings to work with.

In [4]:
# Load dataset using DeepChem
tasks, datasets, transformers = dc.molnet.load_muv(splitter='stratified')
train_dataset, valid_dataset, test_dataset = datasets
train_smiles = train_dataset.ids
valid_smiles = valid_dataset.ids

[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerator
[05:30:50] DEPRECATION WARNING: please use MorganGenerat

We need to define the "alphabet" for our `SeqToSeq` model, the list of all tokens that can appear in sequences.  (It's also possible for input and output sequences to have different alphabets, but since we're training it as an autoencoder, they're identical in this case.)  Make a list of every character that appears in any training sequence.

In [5]:
# Create tokens from both train_smiles and valid_smiles
train_smiles = list(train_smiles)  # Convert to list if it's a numpy array
valid_smiles = list(valid_smiles)  # Convert to list if it's a numpy array

# Combine both datasets to extract tokens
tokens = set()
for s in train_smiles + valid_smiles:
    tokens = tokens.union(set(c for c in s))
tokens = sorted(list(tokens))

Create the model and define the optimization method to use.  In this case, learning works much better if we gradually decrease the learning rate.  We use an `ExponentialDecay` to multiply the learning rate by 0.9 after each epoch.

In [6]:
# class VariationalRandomizer(Layer):
#     """Add random noise to the embedding and include a corresponding loss."""

#     def __init__(self, embedding_dimension, annealing_start_step,
#                  annealing_final_step, **kwargs):
#         super(VariationalRandomizer, self).__init__(**kwargs)
#         self._embedding_dimension = embedding_dimension
#         self._annealing_final_step = annealing_final_step
#         self._annealing_start_step = annealing_start_step
#         self.dense_mean = Dense(embedding_dimension)
#         self.dense_stddev = Dense(embedding_dimension)
#         self.combine = layers.CombineMeanStd(training_only=True)

#     def call(self, inputs, training=True):
#         input, global_step = inputs
#         embedding_mean = self.dense_mean(input)
#         embedding_stddev = self.dense_stddev(input)
#         embedding = self.combine([embedding_mean, embedding_stddev],
#                                  training=training)
#         mean_sq = embedding_mean * embedding_mean
#         stddev_sq = embedding_stddev * embedding_stddev
#         kl = mean_sq + stddev_sq - tf.math.log(stddev_sq + 1e-20) - 1
#         anneal_steps = self._annealing_final_step - self._annealing_start_step
#         if anneal_steps > 0:
#             current_step = tf.cast(global_step,
#                                    tf.float32) - self._annealing_start_step
#             anneal_frac = tf.maximum(0.0, current_step) / anneal_steps
#             kl_scale = tf.minimum(1.0, anneal_frac * anneal_frac)
#         else:
#             kl_scale = 1.0
#         self.add_loss(0.5 * kl_scale * tf.reduce_mean(kl))
#         return embedding

In [7]:
# # TensorFlow
# class SeqToSeq(KerasModel):
#   """Implements sequence to sequence translation models.

#   The model is based on the description in Sutskever et al., "Sequence to
#   Sequence Learning with Neural Networks" (https://arxiv.org/abs/1409.3215),
#   although this implementation uses GRUs instead of LSTMs.  The goal is to
#   take sequences of tokens as input, and translate each one into a different
#   output sequence.  The input and output sequences can both be of variable
#   length, and an output sequence need not have the same length as the input
#   sequence it was generated from.  For example, these models were originally
#   developed for use in natural language processing.  In that context, the
#   input might be a sequence of English words, and the output might be a
#   sequence of French words.  The goal would be to train the model to translate
#   sentences from English to French.

#   The model consists of two parts called the "encoder" and "decoder".  Each one
#   consists of a stack of recurrent layers.  The job of the encoder is to
#   transform the input sequence into a single, fixed length vector called the
#   "embedding".  That vector contains all relevant information from the input
#   sequence.  The decoder then transforms the embedding vector into the output
#   sequence.

#   These models can be used for various purposes.  First and most obviously,
#   they can be used for sequence to sequence translation.  In any case where you
#   have sequences of tokens, and you want to translate each one into a different
#   sequence, a SeqToSeq model can be trained to perform the translation.

#   Another possible use case is transforming variable length sequences into
#   fixed length vectors.  Many types of models require their inputs to have a
#   fixed shape, which makes it difficult to use them with variable sized inputs
#   (for example, when the input is a molecule, and different molecules have
#   different numbers of atoms).  In that case, you can train a SeqToSeq model as
#   an autoencoder, so that it tries to make the output sequence identical to the
#   input one.  That forces the embedding vector to contain all information from
#   the original sequence.  You can then use the encoder for transforming
#   sequences into fixed length embedding vectors, suitable to use as inputs to
#   other types of models.

#   Another use case is to train the decoder for use as a generative model.  Here
#   again you begin by training the SeqToSeq model as an autoencoder.  Once
#   training is complete, you can supply arbitrary embedding vectors, and
#   transform each one into an output sequence.  When used in this way, you
#   typically train it as a variational autoencoder.  This adds random noise to
#   the encoder, and also adds a constraint term to the loss that forces the
#   embedding vector to have a unit Gaussian distribution.  You can then pick
#   random vectors from a Gaussian distribution, and the output sequences should
#   follow the same distribution as the training data.

#   When training as a variational autoencoder, it is best to use KL cost
#   annealing, as described in https://arxiv.org/abs/1511.06349.  The constraint
#   term in the loss is initially set to 0, so the optimizer just tries to
#   minimize the reconstruction loss.  Once it has made reasonable progress
#   toward that, the constraint term can be gradually turned back on.  The range
#   of steps over which this happens is configurable.
#   """

#   sequence_end = object()

#   def __init__(self,
#                input_tokens,
#                output_tokens,
#                max_output_length,
#                encoder_layers=4,
#                decoder_layers=4,
#                embedding_dimension=512,
#                dropout=0.0,
#                reverse_input=True,
#                variational=False,
#                annealing_start_step=5000,
#                annealing_final_step=10000,
#                **kwargs):
#     """Construct a SeqToSeq model.

#     In addition to the following arguments, this class also accepts all the keyword arguments
#     from TensorGraph.

#     Parameters
#     ----------
#     input_tokens: list
#       a list of all tokens that may appear in input sequences
#     output_tokens: list
#       a list of all tokens that may appear in output sequences
#     max_output_length: int
#       the maximum length of output sequence that may be generated
#     encoder_layers: int
#       the number of recurrent layers in the encoder
#     decoder_layers: int
#       the number of recurrent layers in the decoder
#     embedding_dimension: int
#       the width of the embedding vector.  This also is the width of all
#       recurrent layers.
#     
# 
# 
# : float
#       the dropout probability to use during training
#     reverse_input: bool
#       if True, reverse the order of input sequences before sending them into
#       the encoder.  This can improve performance when working with long sequences.
#     variational: bool
#       if True, train the model as a variational autoencoder.  This adds random
#       noise to the encoder, and also constrains the embedding to follow a unit
#       Gaussian distribution.
#     annealing_start_step: int
#       the step (that is, batch) at which to begin turning on the constraint term
#       for KL cost annealing
#     annealing_final_step: int
#       the step (that is, batch) at which to finish turning on the constraint term
#       for KL cost annealing
#     """
#     if SeqToSeq.sequence_end not in input_tokens:
#       input_tokens = input_tokens + [SeqToSeq.sequence_end]
#     if SeqToSeq.sequence_end not in output_tokens:
#       output_tokens = output_tokens + [SeqToSeq.sequence_end]
#     self._input_tokens = input_tokens
#     self._output_tokens = output_tokens
#     self._input_dict = dict((x, i) for i, x in enumerate(input_tokens))
#     self._output_dict = dict((x, i) for i, x in enumerate(output_tokens))
#     self._max_output_length = max_output_length
#     self._embedding_dimension = embedding_dimension
#     self._reverse_input = reverse_input
#     self.encoder = self._create_encoder(encoder_layers, dropout)
#     self.decoder = self._create_decoder(decoder_layers, dropout)
    
#     features = self._create_features()
#     gather_indices = Input(shape=(2,), dtype=tf.int32)
#     global_step = Input(shape=tuple(), dtype=tf.int32)
#     embedding = self.encoder([features, gather_indices])
#     self._embedding = self.encoder([features, gather_indices], training=False)
#     if variational:
#       randomizer = VariationalRandomizer(
#           self._embedding_dimension, annealing_start_step, annealing_final_step)
#       embedding = randomizer([self._embedding, global_step])
#       self._embedding = randomizer(
#           [self._embedding, global_step], training=False)
#     output = self.decoder(embedding)
#     model = tf.keras.Model(
#         inputs=[features, gather_indices, global_step], outputs=output)
#     super(SeqToSeq, self).__init__(model, self._create_loss(), **kwargs)

#   def _create_features(self):
#     return Input(shape=(None, len(self._input_tokens)))

#   def _create_encoder(self, n_layers, dropout):
#     """Create the encoder as a tf.keras.Model."""
#     input = self._create_features()
#     gather_indices = Input(shape=(2,), dtype=tf.int32)
#     prev_layer = input
#     for i in range(n_layers):
#       if dropout > 0.0:
#         prev_layer = Dropout(rate=dropout)(prev_layer)
#       prev_layer = GRU(
#           self._embedding_dimension, return_sequences=True)(prev_layer)
#     prev_layer = Lambda(lambda x: tf.gather_nd(x[0], x[1]))(
#         [prev_layer, gather_indices])
#     return tf.keras.Model(inputs=[input, gather_indices], outputs=prev_layer)

#   def _create_decoder(self, n_layers, dropout):
#     """Create the decoder as a tf.keras.Model."""
#     input = Input(shape=(self._embedding_dimension,))
#     prev_layer = layers.Stack()(self._max_output_length * [input])
#     for i in range(n_layers):
#       if dropout > 0.0:
#         prev_layer = Dropout(dropout)(prev_layer)
#       prev_layer = GRU(
#           self._embedding_dimension, return_sequences=True)(prev_layer)
#     output = Dense(
#         len(self._output_tokens), activation=tf.nn.softmax)(prev_layer)
#     return tf.keras.Model(inputs=input, outputs=output)

#   def _create_loss(self):
#     """Create the loss function."""

#     def loss_fn(outputs, labels, weights):
#       prob = tf.reduce_sum(outputs[0] * labels[0], axis=2)
#       mask = tf.reduce_sum(labels[0], axis=2)
#       log_prob = tf.math.log(prob + 1e-20) * mask
#       loss = -tf.reduce_mean(tf.reduce_sum(log_prob, axis=1))
#       return loss + sum(self.model.losses)

#     return loss_fn

#   def fit_sequences(self,
#                     sequences,
#                     max_checkpoints_to_keep=5,
#                     checkpoint_interval=1000,
#                     restore=False):
#     """Train this model on a set of sequences

#     Parameters
#     ----------
#     sequences: iterable
#       the training samples to fit to.  Each sample should be
#       represented as a tuple of the form (input_sequence, output_sequence).
#     max_checkpoints_to_keep: int
#       the maximum number of checkpoints to keep.  Older checkpoints are discarded.
#     checkpoint_interval: int
#       the frequency at which to write checkpoints, measured in training steps.
#     restore: bool
#       if True, restore the model from the most recent checkpoint and continue training
#       from there.  If False, retrain the model from scratch.
#     """
#     self.fit_generator(
#         self._generate_batches(sequences),
#         max_checkpoints_to_keep=max_checkpoints_to_keep,
#         checkpoint_interval=checkpoint_interval,
#         restore=restore)

#   def predict_from_sequences(self, sequences, beam_width=5):
#     """Given a set of input sequences, predict the output sequences.

#     The prediction is done using a beam search with length normalization.

#     Parameters
#     ----------
#     sequences: iterable
#       the input sequences to generate a prediction for
#     beam_width: int
#       the beam width to use for searching.  Set to 1 to use a simple greedy search.
#     """
#     result = []
#     for batch in self._batch_elements(sequences):
#       features = self._create_input_array(batch)
#       indices = np.array([(i, len(batch[i]) if i < len(batch) else 0)
#                           for i in range(self.batch_size)])
#       probs = self.predict_on_generator([[(features, indices,
#                                            np.array(self.get_global_step())),
#                                           None, None]])
#       for i in range(len(batch)):
#         result.append(self._beam_search(probs[i], beam_width))
#     return result

#   def predict_from_embeddings(self, embeddings, beam_width=5):
#     """Given a set of embedding vectors, predict the output sequences.

#     The prediction is done using a beam search with length normalization.

#     Parameters
#     ----------
#     embeddings: iterable
#       the embedding vectors to generate predictions for
#     beam_width: int
#       the beam width to use for searching.  Set to 1 to use a simple greedy search.
#     """
#     result = []
#     for batch in self._batch_elements(embeddings):
#       embedding_array = np.zeros(
#           (self.batch_size, self._embedding_dimension), dtype=np.float32)
#       for i, e in enumerate(batch):
#         embedding_array[i] = e
#       probs = self.decoder(embedding_array, training=False)
#       probs = probs.numpy()
#       for i in range(len(batch)):
#         result.append(self._beam_search(probs[i], beam_width))
#     return result

#   def predict_embeddings(self, sequences):
#     """Given a set of input sequences, compute the embedding vectors.

#     Parameters
#     ----------
#     sequences: iterable
#       the input sequences to generate an embedding vector for
#     """
#     result = []
#     for batch in self._batch_elements(sequences):
#       features = self._create_input_array(batch)
#       indices = np.array([(i, len(batch[i]) if i < len(batch) else 0)
#                           for i in range(self.batch_size)])
#       embeddings = self.predict_on_generator(
#           [[(features, indices, np.array(self.get_global_step())), None, None]],
#           outputs=self._embedding)
#       for i in range(len(batch)):
#         result.append(embeddings[i])
#     return np.array(result, dtype=np.float32)

#   def _beam_search(self, probs, beam_width):
#     """Perform a beam search for the most likely output sequence."""
#     if beam_width == 1:
#       # Do a simple greedy search.

#       s = []
#       for i in range(len(probs)):
#         token = self._output_tokens[np.argmax(probs[i])]
#         if token == SeqToSeq.sequence_end:
#           break
#         s.append(token)
#       return s

#     # Do a beam search with length normalization.

#     logprobs = np.log(probs)
#     # Represent each candidate as (normalized prob, raw prob, sequence)
#     candidates = [(0.0, 0.0, [])]
#     for i in range(len(logprobs)):
#       new_candidates = []
#       for c in candidates:
#         if len(c[2]) > 0 and c[2][-1] == SeqToSeq.sequence_end:
#           # This candidate sequence has already been terminated
#           if len(new_candidates) < beam_width:
#             heappush(new_candidates, c)
#           else:
#             heappushpop(new_candidates, c)
#         else:
#           # Consider all possible tokens we could add to this candidate sequence.
#           for j, logprob in enumerate(logprobs[i]):
#             new_logprob = logprob + c[1]
#             newc = (new_logprob / (len(c[2]) + 1), new_logprob,
#                     c[2] + [self._output_tokens[j]])
#             if len(new_candidates) < beam_width:
#               heappush(new_candidates, newc)
#             else:
#               heappushpop(new_candidates, newc)
#       candidates = new_candidates
#     return sorted(candidates)[-1][2][:-1]

#   def _create_input_array(self, sequences):
#     """Create the array describing the input sequences for a batch."""
#     lengths = [len(x) for x in sequences]
#     if self._reverse_input:
#       sequences = [reversed(s) for s in sequences]
#     features = np.zeros(
#         (self.batch_size, max(lengths) + 1, len(self._input_tokens)),
#         dtype=np.float32)
#     for i, sequence in enumerate(sequences):
#       for j, token in enumerate(sequence):
#         features[i, j, self._input_dict[token]] = 1
#     features[np.arange(len(sequences)), lengths, self._input_dict[
#         SeqToSeq.sequence_end]] = 1
#     return features

#   def _create_output_array(self, sequences):
#     """Create the array describing the target sequences for a batch."""
#     lengths = [len(x) for x in sequences]
#     labels = np.zeros(
#         (self.batch_size, self._max_output_length, len(self._output_tokens)),
#         dtype=np.float32)
#     end_marker_index = self._output_dict[SeqToSeq.sequence_end]
#     for i, sequence in enumerate(sequences):
#       for j, token in enumerate(sequence):
#         labels[i, j, self._output_dict[token]] = 1
#       for j in range(lengths[i], self._max_output_length):
#         labels[i, j, end_marker_index] = 1
#     return labels

#   def _batch_elements(self, elements):
#     """Combine elements into batches."""
#     batch = []
#     for s in elements:
#       batch.append(s)
#       if len(batch) == self.batch_size:
#         yield batch
#         batch = []
#     if len(batch) > 0:
#       yield batcharalla

#   def _generate_batches(self, sequences):
#     """Create feed_dicts for fitting."""
#     for batch in self._batch_elements(sequences):
#       inputs = []
#       outputs = []
#       for input, output in batch:
#         inputs.append(input)
#         outputs.append(output)
#       for i in range(len(inputs), self.batch_size):
#         inputs.append([])
#         outputs.append([])
#       features = self._create_input_array(inputs)
#       labels = self._create_output_array(outputs)
#       gather_indices = np.array([(i, len(x)) for i, x in enumerate(inputs)])
#       yield ([features, gather_indices,
#               np.array(self.get_global_step())], [labels], [])

### Torch model with One-Hot Encoding and Decoding

This SeqToSeq_OneHot model is specifically designed for this tutorial to demonstrate the usage of a SeqToSeq model for generating fingerprints to classify molecules. While DeepChem already provides a SeqToSeq model in PyTorch, that implementation uses index-based encoding and decoding, which is effective for many tasks but unsuitable for reproducing SMILES strings directly from embeddings. The TensorFlow model used for this task employs one-hot encoding and decoding for the SeqToSeq functionality. To align with the TensorFlow model's behavior and meet the requirements of this task, the PyTorch SeqToSeq_OneHot model was created to incorporate one-hot encoding and decoding functionality.

In [8]:
# Refactored PyTorch Seq2Seq with one-hot like Tensorflow one
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from heapq import heappush, heappushpop
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from deepchem.utils.pytorch_utils import get_activation


class TokenManager:
    """
    Manages token vocabularies and conversions between tokens and indices.
    
    This class is essential because sequence-to-sequence models work with discrete tokens
    (like words, characters, or SMILES molecular tokens) but neural networks need numerical
    representations. The TokenManager bridges this gap by:
    
    1. Maintaining separate vocabularies for input and output sequences (they may differ)
    2. Converting tokens to indices (encoding) for neural network processing
    3. Converting indices back to tokens (decoding) for human-readable output
    4. Ensuring both vocabularies contain a special sequence_end token for proper termination
    
    The separation of input/output vocabularies is crucial for tasks like molecular translation
    where input might be one representation (e.g., SMILES) and output another (e.g., fingerprints).
    """

    # Special sentinel object to mark sequence termination 
    sequence_end = object() 
    
    def __init__(self, input_tokens, output_tokens):
        # Ensure sequence_end is in both vocabularies
        if self.sequence_end not in input_tokens:
            input_tokens = input_tokens + [self.sequence_end]
        if self.sequence_end not in output_tokens:
            output_tokens = output_tokens + [self.sequence_end]
            
        self.input_tokens = input_tokens
        self.output_tokens = output_tokens

        # Create bidirectional mappings for O(1) lookup efficiency
        self.input_dict = {x: i for i, x in enumerate(input_tokens)}
        self.output_dict = {x: i for i, x in enumerate(output_tokens)}
    
    @property
    def input_vocab_size(self):
        """Returns size of input vocabulary - needed for embedding layer initialization."""
        return len(self.input_tokens)
    
    @property
    def output_vocab_size(self):
        """Returns size of output vocabulary - needed for output projection layer."""
        return len(self.output_tokens)
    
    def encode_input_token(self, token):
        """
        Convert input token to index with fallback handling.
        
        Uses get() with fallback to handle unknown tokens gracefully - in production,
        you'd typically have a dedicated <UNK> token for out-of-vocabulary words.
        """
        return self.input_dict.get(token, self.input_dict.get('<UNK>', 0))
    
    def encode_output_token(self, token):
        """Convert output token to index with fallback handling."""
        return self.output_dict.get(token, self.output_dict.get('<UNK>', 0))
    
    def decode_output_token(self, index):
        """Convert output index back to token - used during inference to get readable results."""
        return self.output_tokens[index]


class Encoder(nn.Module):
    """
    Multi-layer GRU encoder with embedding layer.
    
    This design follows the classic sequence-to-sequence architecture where:
    1. Token indices are converted to dense embeddings (learned representations)
    2. A multi-layer GRU processes the sequence, building up contextual understanding
    3. The final state represents the entire input sequence as a fixed-size vector
    
    GRU (Gated Recurrent Unit) is chosen over LSTM for computational efficiency while
    still handling long-term dependencies. The multi-layer design allows the model
    to learn hierarchical representations - lower layers might capture syntax,
    higher layers capture semantics.
    
    The gather_indices parameter allows extracting embeddings from specific positions,
    which is useful when sequences have different lengths and you want the embedding
    from the actual end position rather than a padded position.
    """
    
    def __init__(self, 
                 input_vocab_size, 
                 embedding_dim, 
                 n_layers=4, 
                 dropout=0.1,
                 **kwargs):
        super(Encoder, self).__init__(**kwargs)
        self.embedding = nn.Embedding(input_vocab_size, embedding_dim)
        self.embedding_dim = embedding_dim
        self.n_layers = n_layers
        self.gru = nn.GRU(embedding_dim, embedding_dim, n_layers, 
                         batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, input_indices, gather_indices=None):
        """
        Parameters:
        - input_indices: Token indices (batch_size, seq_len)
        - gather_indices: Indices indicating the end of each sequence (batch_size,)
        
        Returns:
        - Embedding vector for each sequence (batch_size, embedding_dim)
        """
        # Convert token indices to dense embeddings
        embedded = self.dropout(self.embedding(input_indices))
        
        # Pass through GRU layers
        output, hidden = self.gru(embedded)
        
        # Extract the final state for each sequence
        if gather_indices is not None:
            # Use gather_indices to get the embedding at the actual sequence end
            # This avoids using padding positions which would give meaningless embeddings
            batch_size = output.size(0)
            batch_indices = torch.arange(batch_size, device=output.device)
            # Clamp indices to avoid out-of-bounds
            gather_indices = torch.clamp(gather_indices, 0, output.size(1) - 1)
            embedding = output[batch_indices, gather_indices]
        else:
            # Use the last hidden state from the final layer
            embedding = hidden[-1]
            
        return embedding


class Decoder(nn.Module):
    """
    Multi-layer GRU decoder with embedding and attention-like mechanism.
    
    The decoder's role is to transform the fixed-size encoder embedding back into
    a variable-length output sequence. This is more complex than encoding because:
    
    1. During training: Uses "teacher forcing" - feeds the correct previous token
       as input to predict the next token. This stabilizes training by providing
       ground truth context rather than potentially wrong model predictions.
    
    2. During inference: Must generate sequentially, using its own predictions
       as input for the next step (autoregressive generation).
    
    The max_output_length parameter prevents infinite generation and controls
    computational cost. In molecular applications, this might correspond to
    maximum molecule size constraints.
    
    The step_activation adds non-linearity during inference steps, which can
    help with gradient flow and model expressiveness during autoregressive generation.
    """
    
    def __init__(self, 
                 embedding_dim, 
                 output_vocab_size, 
                 max_output_length, 
                 n_layers=4, 
                 dropout=0.1, 
                 step_activation:str = "relu",
                **kwargs):
        super(Decoder, self).__init__(**kwargs)
        self.embedding_dim = embedding_dim
        self.max_output_length = max_output_length
        self.n_layers = n_layers
        self.output_vocab_size = output_vocab_size
        self.step_act = get_activation(step_activation)
        
        # Output token embedding (for teacher forcing during training)
        self.output_embedding = nn.Embedding(output_vocab_size, embedding_dim)
        
        # Multi-layer GRU decoder
        self.gru = nn.GRU(embedding_dim, embedding_dim, n_layers, 
                         batch_first=True, dropout=dropout if n_layers > 1 else 0)
        
        # Output projection layer
        self.output_projection = nn.Linear(embedding_dim, output_vocab_size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, encoder_embedding, target_indices=None):
        """
        Parameters:
        - encoder_embedding: Embedding vector from encoder (batch_size, embedding_dim)
        - target_indices: Target token indices for teacher forcing (batch_size, seq_len) [optional]
        
        Returns:
        - Token probabilities for each position (batch_size, max_output_length, output_vocab_size)

        The dual-mode operation (training vs inference) is critical:
        - Training mode uses teacher forcing for stable, parallel training
        - Inference mode generates sequentially for actual use
        """
        batch_size = encoder_embedding.size(0)
        device = encoder_embedding.device
        
        if self.training and target_indices is not None:
            # Teacher forcing mode - use target tokens as input
            return self._forward_teacher_forcing(encoder_embedding, target_indices)
        else:
            # Inference mode - generate sequentially
            return self._forward_inference(encoder_embedding, batch_size, device)
    
    def _forward_teacher_forcing(self, encoder_embedding, target_indices):
        """
        Forward pass with teacher forcing (for training).
        
        Teacher forcing feeds the correct previous token as input to predict the next token.
        This creates a stable training signal because the model always sees correct context,
        rather than its own potentially incorrect predictions. The tradeoff is that there's
        a mismatch between training (teacher forcing) and inference (autoregressive).
        """
        batch_size, seq_len = target_indices.shape
        
        # Embed target tokens (shifted right for teacher forcing)
        # We'll use the encoder embedding as the first input and target tokens for the rest
        embedded_targets = self.dropout(self.output_embedding(target_indices))
        
        # Prepare decoder input: encoder_embedding + embedded_targets
        decoder_input = torch.cat([
            encoder_embedding.unsqueeze(1),  # First input is encoder output
            embedded_targets[:, :-1]  # Shifted target sequence
        ], dim=1)
        
        # Initialize hidden state with encoder embedding
        hidden = encoder_embedding.unsqueeze(0).repeat(self.n_layers, 1, 1)
        
        # Pass through GRU
        output, _ = self.gru(decoder_input, hidden)
        
        # Project to vocabulary
        logits = self.output_projection(output)
        probabilities = F.softmax(logits, dim=2)
        
        return probabilities
    

    def _forward_inference(self, encoder_embedding, batch_size, device):
        """
        Forward pass during inference - generates tokens sequentially.
        
        This is the autoregressive generation process where each token is generated
        based on the previously generated tokens. It's slower than teacher forcing
        because it must generate one token at a time, but it's how the model
        actually works during real use.
        """
        outputs = []
        hidden = encoder_embedding.unsqueeze(0).repeat(self.n_layers, 1, 1)

        # Initialize with START token (or encoder embedding trick)
        input_token = torch.zeros(batch_size, dtype=torch.long, device=device)  # assuming 0 is <sos>

        for i in range(self.max_output_length):
            logits, hidden = self.step(input_token, hidden)  # Use step function
            probabilities = F.softmax(logits, dim=2)
            outputs.append(probabilities)

            # Greedy decoding: pick highest probability
            input_token = torch.argmax(probabilities, dim=2).squeeze(1)  # [batch_size]

        return torch.cat(outputs, dim=1)
    

    def step(self, input_token, hidden):
        """
        Perform a single decoding step.

        This method is the core of autoregressive generation. It takes the current
        token and hidden state, and produces the next token probabilities and
        updated hidden state. The step_activation adds non-linearity which can
        help with gradient flow during backpropagation through the generation process.

        Parameters:
        - input_token: tensor of shape [batch_size] or [batch_size, 1] (token indices)
        - hidden: hidden state from previous step [n_layers, batch_size, embedding_dim]

        Returns:
        - logits: raw output logits [batch_size, 1, vocab_size]
        - hidden: updated hidden state
        """
        if input_token.dim() == 1:
            # shape [batch_size] -> [batch_size, 1]
            input_token = input_token.unsqueeze(1)

        # Embed the input token: [batch_size, 1, embedding_dim]
        embedded = self.output_embedding(input_token)
        embedded = self.step_act(embedded)  # Apply non-linearity

        # No dropout during inference
        output, hidden = self.gru(embedded, hidden)

        # Project to vocab size: [batch_size, 1, vocab_size]
        logits = self.output_projection(output)

        return logits, hidden
    

class VariationalLayer(nn.Module):
    """
    Variational autoencoder layer for adding latent variable modeling.
    
    This layer implements the "variational" part of a Variational Autoencoder (VAE).
    Instead of using deterministic embeddings, it learns a probability distribution
    over embeddings and samples from it. This has several benefits:
    
    1. Regularization: The KL divergence loss encourages the latent space to be
       well-structured and prevents overfitting to specific embeddings.
    
    2. Generative capability: You can sample random embeddings from the learned
       distribution to generate new sequences that follow the training data distribution.
    
    3. Smooth latent space: Similar inputs produce similar distributions, creating
       a continuous latent space good for interpolation and exploration.
    
    The annealing schedule is crucial for VAE training. If you apply full KL loss
    from the start, the model might collapse to the prior (unit Gaussian) and ignore
    the input data. Annealing gradually increases the KL weight, allowing the model
    to first learn to reconstruct, then learn a good latent structure.
    """
    
    def __init__(self, embedding_dim, annealing_start_step=5000, annealing_final_step=10000):
        super(VariationalLayer, self).__init__()
        self.mu_fc = nn.Linear(embedding_dim, embedding_dim)
        self.logvar_fc = nn.Linear(embedding_dim, embedding_dim)
        self.annealing_start_step = annealing_start_step
        self.annealing_final_step = annealing_final_step
        self.global_step = 0
        self.kl_loss = 0
    
    def forward(self, embedding):
        """
        Apply variational sampling to embeddings.

        Parameters:
        - embedding: Input embedding (batch_size, embedding_dim)
        
        Returns:
        - Modified embedding with variational sampling

        The reparameterization trick is essential for backpropagation through
        stochastic nodes. Instead of sampling directly from N(μ, σ²), we sample
        ε ~ N(0,1) and compute μ + σ·ε. This makes the sampling operation
        differentiable with respect to μ and σ.
        """
        if self.training:
            mu = self.mu_fc(embedding)
            logvar = self.logvar_fc(embedding)
            
            # Reparameterization trick
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            embedding = mu + eps * std
            
            # Calculate KL divergence loss
            kl_weight = self._get_kl_weight()
            kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            self.kl_loss = kl_weight * kl_loss
        else:
            self.kl_loss = 0
            
        return embedding
    
    def _get_kl_weight(self):
        """
        Calculate the KL weight for annealing.
        
        Annealing is critical for VAE training success. Without it, the model
        might ignore the input and just learn to sample from the prior.
        The schedule gradually increases KL weight from 0 to 1, allowing
        the model to first learn reconstruction, then regularization.
        """
        if self.global_step < self.annealing_start_step:
            return 0.0
        elif self.global_step > self.annealing_final_step:
            return 1.0
        else:
            return (self.global_step - self.annealing_start_step) / (
                self.annealing_final_step - self.annealing_start_step)


class BeamSearchDecoder:
    """
    Handles beam search decoding for sequence generation.
    
    Beam search is a heuristic search algorithm that explores multiple candidate
    sequences simultaneously, keeping only the top-k most promising candidates
    at each step. This is superior to greedy search (which only keeps the single
    best candidate) because:
    
    1. It can recover from locally suboptimal choices
    2. It considers multiple possible continuations
    3. It typically produces higher-quality sequences than greedy search
    
    The algorithm maintains a "beam" of candidate sequences, expanding each one
    with all possible next tokens, then keeping only the best candidates.
    Length normalization prevents the algorithm from favoring shorter sequences
    (since log probabilities are negative and longer sequences accumulate more
    negative values).
    """
    
    def __init__(self, token_manager):
        self.token_manager = token_manager
    
    def decode(self, probs, beam_width=5):
        """
        Perform beam search for the most likely output sequence.
        
        Parameters:
        - probs: Output token probabilities (seq_len, vocab_size)
        - beam_width: Width of the beam search
        
        Returns:
        - The most likely output sequence
        """
        if beam_width == 1:
            return self._greedy_search(probs)
        
        return self._beam_search(probs, beam_width)
    
    def _greedy_search(self, probs):
        """Simple greedy search - always pick the highest probability token."""
        sequence = []
        for i in range(len(probs)):
            token_idx = np.argmax(probs[i])
            token = self.token_manager.decode_output_token(token_idx)

            if token == self.token_manager.sequence_end:
                break
            sequence.append(token)
        return sequence
    
    def _beam_search(self, probs, beam_width):
        """
        Beam search with length normalization.
        
        The algorithm maintains candidates as (normalized_prob, raw_prob, sequence)
        tuples. Length normalization prevents bias toward shorter sequences since
        log probabilities are negative and longer sequences accumulate more negative values.
        
        Uses heap data structure for efficient top-k operations.
        """
        # Convert to log probabilities for numerical stability
        # Add small epsilon to prevent log(0)
        logprobs = np.log(probs + 1e-20)

        # Represent each candidate as (normalized prob, raw prob, sequence)
        # Start with empty sequence and zero probability
        candidates = [(0.0, 0.0, [])]
        
        # Expand beam for each position in the sequence
        for i in range(len(logprobs)):
            new_candidates = []

            # Expand each existing candidate
            for c in candidates:
                if (len(c[2]) > 0 and 
                    c[2][-1] == self.token_manager.sequence_end):
                    # Add terminated sequence to new candidates
                    if len(new_candidates) < beam_width:
                        heappush(new_candidates, c)
                    else:
                        heappushpop(new_candidates, c)
                else:
                    # Consider all possible tokens
                    for j, logprob in enumerate(logprobs[i]):
                        new_logprob = logprob + c[1]
                        token = self.token_manager.decode_output_token(j)

                        # Create new candidate with length normalization
                        # Divide by sequence length to prevent bias toward shorter sequences
                        newc = (new_logprob / (len(c[2]) + 1), new_logprob,
                                c[2] + [token])
                        
                        # Maintain beam_width best candidates using heap
                        if len(new_candidates) < beam_width:
                            heappush(new_candidates, newc)
                        else:
                            heappushpop(new_candidates, newc)

            candidates = new_candidates
        
        # Return the best sequence (highest normalized probability)
        # Remove sequence_end token if present for cleaner output
        best_sequence = sorted(candidates)[-1][2]
        if (best_sequence and 
            best_sequence[-1] == self.token_manager.sequence_end):
            return best_sequence[:-1]
        return best_sequence


class DataProcessor:
    """
    Handles data preprocessing and batch creation for index-based inputs.
    
    This class bridges the gap between raw token sequences and the numerical
    tensors that neural networks require. Key responsibilities:
    
    1. Converting token sequences to index sequences using TokenManager
    2. Handling variable-length sequences through padding
    3. Implementing sequence reversal (helps with sequence-to-sequence learning)
    4. Creating batches for efficient GPU processing
    5. Generating one-hot encodings for compatibility with TensorFlow models
    
    Sequence reversal is a technique from the original seq2seq paper that can
    improve learning by creating shorter dependency paths between input and
    output tokens. The idea is that the first tokens of input and output are
    now closer in the computational graph.
    
    The batch processing is essential for GPU efficiency - processing sequences
    one at a time would be extremely slow compared to batch processing.
    """
    
    def __init__(self, token_manager, max_output_length, reverse_input=True, batch_size=32):
        self.token_manager = token_manager
        self.max_output_length = max_output_length
        self.reverse_input = reverse_input
        self.batch_size = batch_size
    
    def create_input_array(self, sequences):
        """Create token index array for a batch of sequences."""
        # Convert sequences to token indices
        indexed_sequences = []
        for sequence in sequences:
            if self.reverse_input:
                sequence = list(reversed(sequence))
            
            # Convert tokens to indices and add sequence_end
            indices = [self.token_manager.encode_input_token(token) for token in sequence]
            indices.append(self.token_manager.encode_input_token(self.token_manager.sequence_end))
            indexed_sequences.append(indices)
        
        # Pad sequences to same length
        max_len = max(len(seq) for seq in indexed_sequences)
        padded_sequences = []
        
        for seq in indexed_sequences:
            padded_seq = seq + [0] * (max_len - len(seq))  # Pad with 0 (assuming 0 is padding)
            padded_sequences.append(padded_seq)
        
        return torch.tensor(padded_sequences, dtype=torch.long)
    
    def create_output_array(self, sequences):
        """Create token index array for target sequences."""
        indexed_sequences = []
        
        for sequence in sequences:
            # Convert tokens to indices, truncate if too long
            indices = [self.token_manager.encode_output_token(token) 
                      for token in sequence[:self.max_output_length-1]]
            # Add sequence_end token
            indices.append(self.token_manager.encode_output_token(self.token_manager.sequence_end))
            
            # Pad to max_output_length
            while len(indices) < self.max_output_length:
                indices.append(self.token_manager.encode_output_token(self.token_manager.sequence_end))
            
            indexed_sequences.append(indices)
        
        return torch.tensor(indexed_sequences, dtype=torch.long)
    
    def create_output_one_hot(self, sequences):
        """Create one-hot encoded target array (for loss calculation)."""
        target_indices = self.create_output_array(sequences)
        batch_size, seq_len = target_indices.shape
        
        # Create one-hot encoding
        one_hot = torch.zeros(batch_size, seq_len, self.token_manager.output_vocab_size)
        one_hot.scatter_(2, target_indices.unsqueeze(2), 1)
        
        return one_hot
    
    def batch_elements(self, elements, batch_size=None):
        """Combine elements into batches."""
        if batch_size is None:
            batch_size = self.batch_size
        
        batch = []
        for element in elements:
            batch.append(element)
            if len(batch) == batch_size:
                yield batch
                batch = []
        if len(batch) > 0:
            yield batch


class SeqToSeq_OneHot(nn.Module):
    """
    This SeqToSeq_OneHot model is specifically designed for this tutorial to 
    demonstrate the usage of a SeqToSeq model for generating fingerprints to 
    classify molecules. While DeepChem already provides a SeqToSeq model in 
    PyTorch, that implementation uses index-based encoding and decoding, which 
    is effective for many tasks but unsuitable for reproducing SMILES strings 
    directly from embeddings. The TensorFlow model used for this task employs 
    one-hot encoding and decoding for the SeqToSeq functionality. To align 
    with the TensorFlow model's behavior and meet the requirements of this 
    task, the PyTorch SeqToSeq_OneHot model was created to incorporate one-hot 
    encoding and decoding functionality. 

    (Docstring from Tensorflow model)
    Implements sequence to sequence translation models.

    The model is based on the description in Sutskever et al., "Sequence to
    Sequence Learning with Neural Networks" (https://arxiv.org/abs/1409.3215),
    although this implementation uses GRUs instead of LSTMs.  The goal is to
    take sequences of tokens as input, and translate each one into a different
    output sequence.  The input and output sequences can both be of variable
    length, and an output sequence need not have the same length as the input
    sequence it was generated from.  For example, these models were originally
    developed for use in natural language processing.  In that context, the
    input might be a sequence of English words, and the output might be a
    sequence of French words.  The goal would be to train the model to translate
    sentences from English to French.

    The model consists of two parts called the "encoder" and "decoder".  Each one
    consists of a stack of recurrent layers.  The job of the encoder is to
    transform the input sequence into a single, fixed length vector called the
    "embedding".  That vector contains all relevant information from the input
    sequence.  The decoder then transforms the embedding vector into the output
    sequence.

    These models can be used for various purposes.  First and most obviously,
    they can be used for sequence to sequence translation.  In any case where you
    have sequences of tokens, and you want to translate each one into a different
    sequence, a SeqToSeq model can be trained to perform the translation.

    Another possible use case is transforming variable length sequences into
    fixed length vectors.  Many types of models require their inputs to have a
    fixed shape, which makes it difficult to use them with variable sized inputs
    (for example, when the input is a molecule, and different molecules have
    different numbers of atoms).  In that case, you can train a SeqToSeq model as
    an autoencoder, so that it tries to make the output sequence identical to the
    input one.  That forces the embedding vector to contain all information from
    the original sequence.  You can then use the encoder for transforming
    sequences into fixed length embedding vectors, suitable to use as inputs to
    other types of models.

    Another use case is to train the decoder for use as a generative model.  Here
    again you begin by training the SeqToSeq model as an autoencoder.  Once
    training is complete, you can supply arbitrary embedding vectors, and
    transform each one into an output sequence.  When used in this way, you
    typically train it as a variational autoencoder.  This adds random noise to
    the encoder, and also adds a constraint term to the loss that forces the
    embedding vector to have a unit Gaussian distribution.  You can then pick
    random vectors from a Gaussian distribution, and the output sequences should
    follow the same distribution as the training data.

    When training as a variational autoencoder, it is best to use KL cost
    annealing, as described in https://arxiv.org/abs/1511.06349.  The constraint
    term in the loss is initially set to 0, so the optimizer just tries to
    minimize the reconstruction loss.  Once it has made reasonable progress
    toward that, the constraint term can be gradually turned back on.  The range
    of steps over which this happens is configurable.
    """
    
    def __init__(self,
                 input_tokens,
                 output_tokens,
                 max_output_length,
                 encoder_layers=4,
                 decoder_layers=4,
                 batch_size=100,
                 embedding_dimension=512,
                 dropout=0.1,
                 reverse_input=True,
                 variational=False,
                 annealing_start_step=5000,
                 annealing_final_step=10000):
        super(SeqToSeq_OneHot, self).__init__()
        
        # Initialize components
        self.token_manager = TokenManager(input_tokens, output_tokens)
        self.data_processor = DataProcessor(
            self.token_manager, max_output_length, reverse_input, batch_size)
        self.beam_decoder = BeamSearchDecoder(self.token_manager)
        
        # Model parameters
        self.max_output_length = max_output_length
        self.embedding_dimension = embedding_dimension
        self.variational = variational
        
        # Create encoder and decoder
        self.encoder = Encoder(
            self.token_manager.input_vocab_size,
            embedding_dimension,
            encoder_layers,
            dropout
        )
        
        self.decoder = Decoder(
            embedding_dimension,
            self.token_manager.output_vocab_size,
            max_output_length,
            decoder_layers,
            dropout
        )
        
        # Add variational layer if requested
        if variational:
            self.variational_layer = VariationalLayer(
                embedding_dimension, annealing_start_step, annealing_final_step)
    
    def encode(self, input_indices, gather_indices=None):
        """Encode input sequences to embeddings."""
        return self.encoder(input_indices, gather_indices)
    
    def decode(self, embedding, target_indices=None):
        """Decode embeddings to output sequences."""
        return self.decoder(embedding, target_indices)
    
    def forward(self, input_indices, gather_indices=None, target_indices=None):
        """Full forward pass through the model."""
        embedding = self.encode(input_indices, gather_indices)
        
        # Apply variational layer if enabled
        if self.variational and hasattr(self, 'variational_layer'):
            embedding = self.variational_layer(embedding)
            self.kl_loss = self.variational_layer.kl_loss
        else:
            self.kl_loss = 0
        
        output = self.decode(embedding, target_indices)
        return output
    
    
    def loss_fn(self, outputs, labels, weights=None): # FIXME: uses unnecesary one-hot (legacy)
        """
        Calculate the loss for a batch.
        
        Parameters:
        - outputs: Predicted token probabilities (batch_size, seq_len, vocab_size)
        - labels: Target token indices (batch_size, seq_len) or one-hot (batch_size, seq_len, vocab_size)
        - weights: Optional weights for each element
        
        Returns:
        - Loss value
        """
        if labels.dim() == 2:  # If labels are indices
            # Convert to one-hot for compatibility
            batch_size, seq_len = labels.shape
            one_hot_labels = torch.zeros(batch_size, seq_len, outputs.size(2), device=labels.device)
            one_hot_labels.scatter_(2, labels.unsqueeze(2), 1)
            labels = one_hot_labels
        
        # Sum the probabilities of the correct tokens
        prob = torch.sum(outputs * labels, dim=2)
        
        # Create a mask to identify valid positions (non-padding)
        mask = torch.sum(labels, dim=2)
        
        # Calculate log probabilities with epsilon to avoid log(0)
        log_prob = torch.log(prob + 1e-20) * mask
        
        # Calculate mean negative log likelihood
        loss = -torch.mean(torch.sum(log_prob, dim=1))
        
        # Add KL divergence loss if using variational autoencoder
        if hasattr(self, 'kl_loss') and self.kl_loss != 0:
            loss = loss + self.kl_loss
        
        return loss
    
    
    def predict_from_sequences(self, sequences, beam_width=5):
        """Predict output sequences from input sequences using beam search."""
        self.eval()
        result = []
        
        with torch.no_grad():
            for batch in self.data_processor.batch_elements(sequences):
                input_indices = self.data_processor.create_input_array(batch)
                device = next(self.parameters()).device
                input_indices = input_indices.to(device)
                
                # Calculate sequence lengths (excluding padding)
                lengths = []
                for seq in batch:
                    length = len(seq) if not self.data_processor.reverse_input else len(seq)
                    lengths.append(length)  # Position of sequence_end token
                lengths = torch.tensor(lengths, device=device)
                
                # Get output probabilities
                probs = self.forward(input_indices, lengths).cpu().numpy()
                
                # Perform beam search for each sequence in the batch
                for i in range(len(batch)):
                    decoded_seq = self.beam_decoder.decode(probs[i], beam_width)
                    result.append(decoded_seq)
        
        return result
    
    def predict_embeddings(self, sequences):
        """Compute embedding vectors for input sequences."""
        self.eval()
        result = []
        
        with torch.no_grad():
            for batch in self.data_processor.batch_elements(sequences):
                input_indices = self.data_processor.create_input_array(batch)
                device = next(self.parameters()).device
                input_indices = input_indices.to(device)
                
                # Calculate sequence lengths
                lengths = []
                for seq in batch:
                    length = len(seq) if not self.data_processor.reverse_input else len(seq)
                    lengths.append(length)
                lengths = torch.tensor(lengths, device=device)
                
                # Get embeddings
                embeddings = self.encode(input_indices, lengths).cpu().numpy()
                for i in range(len(batch)):
                    result.append(embeddings[i])
        
        return np.array(result, dtype=np.float32)
    
    def step_variational_annealing(self):
        """Step the variational annealing schedule."""
        if self.variational and hasattr(self, 'variational_layer'):
            self.variational_layer.global_step += 1
    
    # Backward compatibility methods - delegate to data_processor
    def _batch_elements(self, elements, batch_size=None):
        """Backward compatibility: delegate to data_processor."""
        return self.data_processor.batch_elements(elements, batch_size)
    
    def _create_input_array(self, sequences):
        """Backward compatibility: delegate to data_processor."""
        return self.data_processor.create_input_array(sequences)
    
    def _create_output_array(self, sequences):
        """Backward compatibility: delegate to data_processor (returns one-hot for compatibility)."""
        return self.data_processor.create_output_one_hot(sequences)

# Differences with TensorFlow implementation

The TensorFlow Seq2Seq uses one-hot encoded labels, whereas the DeepChem PyTorch SeqToSeqModel uses indices. One-hot encoding is necessary to reconstruct SMILES strings from the embedded vectors, which is why the PyTorch SeqToSeq_OneHot model was created. However, using indices works well for classification tasks.

Below is a comparison of the two PyTorch models, i.e. PyTorch SeqToSeqModel (indices based) vs PyTorch SeqToSeq_OneHot (one-hot based).

### Torch Model from model.torch_models

In [9]:
from deepchem.models.torch_models.seqtoseq import SeqToSeqModel

print(SeqToSeqModel.__bases__) 

(<class 'deepchem.models.torch_models.torch_model.TorchModel'>,)


In [ ]:
# # Test example

# # Define the dataset
# data_example = [
#     ("Cc1cccc(N2CCN(C(=O)C34CC5CC(CC(C5)C3)C4)CC2)c1C",
#      "Cc1cccc(N2CCN(C(=O)C34CC5CC(CC(C5)C3)C4)CC2)c1C"),
#     ("Cn1ccnc1SCC(=O)Nc1ccc(Oc2ccccc2)cc1",
#      "Cn1ccnc1SCC(=O)Nc1ccc(Oc2ccccc2)cc1"),
#     ("COc1cc2c(cc1NC(=O)CN1C(=O)NC3(CCc4ccccc43)C1=O)oc1ccccc12",
#      "COc1cc2c(cc1NC(=O)CN1C(=O)NC3(CCc4ccccc43)C1=O)oc1ccccc12"),
#     ("O=C1/C(=C/NC2CCS(=O)(=O)C2)c2ccccc2C(=O)N1c1ccccc1",
#      "O=C1/C(=C/NC2CCS(=O)(=O)C2)c2ccccc2C(=O)N1c1ccccc1"),
#     ("NC(=O)NC(Cc1ccccc1)C(=O)O",
#      "NC(=O)NC(Cc1ccccc1)C(=O)O")
# ]

# # Extract SMILES strings and tokens
# train_smiles_example = [s[0] for s in data_example]
# tokens_example = sorted(set(c for s in train_smiles_example for c in s))

# # Define model parameters
# max_length_example = max(len(s) for s in train_smiles_example)
# batch_size_example = 100
# batches_per_epoch_example = len(train_smiles_example) / batch_size_example

# # Initialize the SeqToSeqModel
# model_ind = SeqToSeqModel(
#     input_tokens=tokens_example,
#     output_tokens=tokens_example,
#     max_output_length=max_length_example,
#     encoder_layers=2,
#     decoder_layers=2,
#     embedding_dimension=256,
#     model_dir="fingerprint",
#     batch_size=batch_size_example,
#     learning_rate=ExponentialDecay(0.001, 0.9, batches_per_epoch_example)
# )

# print(f"Training SeqToSeqModel")
# print("------------------------------")

# # Train the model
# for epoch in range(20):
#     loss_ind = model_ind.fit_sequences(data_example)
#     print(f"Epoch {epoch + 1}, Loss: {loss_ind}")

# # Predict sequences
# predictions_ind = model_ind.predict_from_sequences(train_smiles_example, beam_width=5)
# print("Predictions SeqToSeqModel:")
# for smile_ind, prediction_ind in zip(train_smiles_example, predictions_ind):
#     print(f"Input: {smile_ind}, Prediction: {''.join(prediction_ind)}")

In [ ]:
# # Test example

# # Initialize the SeqToSeq_OneHot
# model_oh = SeqToSeq_OneHot(
#     input_tokens=tokens_example,
#     output_tokens=tokens_example,
#     max_output_length=max_length_example,
#     encoder_layers=2,
#     decoder_layers=2,
#     embedding_dimension=256,
#     dropout=0.1,
#     reverse_input=True,
#     variational=False
# )
# model_oh.to(device)

# # Define optimizer and scheduler
# optimizer = Adam(model_oh.parameters(), lr=0.001)
# scheduler = ExponentialLR(optimizer, gamma=0.9)

# # Compile the model for optimization (if using PyTorch 2.0+)
# model_oh = torch.compile(model_oh)

# print(f"Training SeqToSeq_OneHot")
# print("------------------------------")


# # Training loop
# model_oh.train()
# for epoch in range(20):
#     total_loss = 0
#     num_batches = 0
    
#     # Use your own batching logic
#     for batch in model_oh._batch_elements(train_smiles_example, batch_size=batch_size_example):
#         features = model_oh._create_input_array(batch).to(device, non_blocking=True)
#         labels = model_oh._create_output_array(batch).to(device, non_blocking=True)
#         lengths = torch.tensor([len(seq) for seq in batch], dtype=torch.long).to(device, non_blocking=True)

#         optimizer.zero_grad()

#         # Forward pass
#         outputs = model_oh(features, gather_indices=lengths)
#         loss = model_oh.loss_fn(outputs, labels)
        
#         # Backward pass and optimization
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()
#         num_batches += 1

#     scheduler.step()

#     print(f"Epoch {epoch + 1}/{20}, Loss: {total_loss / num_batches:.4f}")


# # Predict sequences
# model_oh.eval()
# predictions_oh = model_oh.predict_from_sequences(train_smiles_example, beam_width=5)
# print("Predictions SeqToSeq_OneHot:")
# for smile_oh, prediction_oh in zip(train_smiles_example, predictions_oh):
#     print(f"Input: {smile_oh}, Prediction: {''.join(prediction_oh)}")

In [ ]:
# Define model parameters
max_length = max(len(s) for s in train_smiles)
batch_size = 100  # started with 100. Increase to speed up convergence
batches_per_epoch = len(train_smiles) / batch_size
embedding_dimension = 256
encoder_layers = 2
decoder_layers = 2
learning_rate = 0.001  # for Adam = 0.001
decay_rate = 0.9
epochs = 40  # original is 40


AttributeError: 'SeqToSeqModel' object has no attribute 'parameters'

In [ ]:
# Initialize the DC SeqToSeq model
model_ind = SeqToSeqModel(
    input_tokens=tokens,
    output_tokens=tokens,
    max_output_length=max_length,
    encoder_layers=encoder_layers,
    decoder_layers=decoder_layers,
    embedding_dimension=embedding_dimension,
    model_dir="fingerprint",
    batch_size=batch_size,
    learning_rate=dc_ExponentialDecay(0.001, 0.9, batches_per_epoch)
)
print(model_ind.device)

In [ ]:
# Initialize the SeqToSeq_OneHot model
model_oh = SeqToSeq_OneHot(
    input_tokens=tokens,
    output_tokens=tokens,
    max_output_length=max_length,
    encoder_layers=encoder_layers,
    decoder_layers=decoder_layers,
    embedding_dimension=embedding_dimension,
    dropout=0.1,
    reverse_input=True,
    variational=False
)

# Move the model to GPU if available
model_oh.to(device)

optimizer_oh = Adam(model_oh.parameters(), lr=learning_rate)
# optimizer_oh = AdamW(model_oh.parameters(), lr=learning_rate, weight_decay=1e-4)
# scheduler_oh = ExponentialLR(optimizer_oh, gamma=decay_rate)
scheduler_oh = OneCycleLR(
    optimizer_oh,
    max_lr=0.01,  # Peak learning rate
    epochs=epochs,
    steps_per_epoch=len(train_smiles) // batch_size,
    pct_start=0.1,  # Quick warmup
    anneal_strategy='cos'
)


Let's train it!  The input to `fit_sequences()` is a generator that produces input/output pairs.  On a good GPU, this should take a few hours or less.

In [ ]:
# SeqToSeqModel
def generate_sequences(epochs):
    for i in range(epochs):
        for s in train_smiles:
            yield (s, s)

# time training
start_time = time.time()

# Train the model
model_ind.fit_sequences(generate_sequences(epochs))

# End timing
end_time = time.time()

# Print total training time
total_time = end_time - start_time
print(f"Total training time: {total_time/60:.2f} minutes")

Total training time: 56.74 minutes


In [ ]:
# SeqToSeq_OneHot

# Compile the model for optimization (if using PyTorch 2.0+)
model_oh = torch.compile(model_oh)

# time training
start_time = time.time()

# Training loop
model_oh.train()
for epoch in range(epochs):
    total_loss = 0
    num_batches = 0
    
    # Use your own batching logic
    for batch in model_oh._batch_elements(train_smiles, batch_size=batch_size):

        features = model_oh._create_input_array(batch).to(device, non_blocking=True)
        labels = model_oh._create_output_array(batch).to(device, non_blocking=True)
        lengths = torch.tensor([len(seq) for seq in batch], dtype=torch.long).to(device, non_blocking=True)

        optimizer_oh.zero_grad()

        # Forward pass
        outputs = model_oh(features, gather_indices=lengths)
        loss = model_oh.loss_fn(outputs, labels)
        
        # Backward pass and optimisation
        loss.backward()
        optimizer_oh.step()

        total_loss += loss.item()
        num_batches += 1

    scheduler_oh.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / num_batches:.4f}")


# End timing
end_time = time.time()

# Print total training time
total_time = end_time - start_time
print(f"Total training time: {total_time/60:.2f} minutes")


Epoch 5/40, Loss: 292.3587
Epoch 10/40, Loss: 292.3557
Epoch 15/40, Loss: 292.3549
Epoch 20/40, Loss: 292.3566
Epoch 25/40, Loss: 292.3538
Epoch 30/40, Loss: 292.3570
Epoch 35/40, Loss: 292.3562
Epoch 40/40, Loss: 292.3578
Total training time: 64.33 minutes


Let's see how well it works as an autoencoder.  We'll run the first 500 molecules from the validation set through it, and see how many of them are exactly reproduced.

In [ ]:
# SeqToSeqModel (indices)

predicted = model_ind.predict_from_sequences(valid_smiles[:500])
count = 0
for s,p in zip(valid_smiles[:500], predicted):
  if ''.join(p) == s:
    count += 1
print(f'SeqToSeqModel: Reproduced {count} of 500 validation SMILES strings')

SeqToSeqModel: Reproduced 52 of 500 validation SMILES strings


In [ ]:
# SeqToSeq_OneHot

# Ensure max_length matches the longest sequence
max_length = max(max(len(s) for s in train_smiles), max(len(s) for s in valid_smiles))
model_oh._max_output_length = max_length  # Update the model_oh's max_output_length

# Truncate validation SMILES to match max_output_length
valid_smiles = [s[:model_oh._max_output_length] for s in valid_smiles]

# Predict sequences
predicted_oh = model_oh.predict_from_sequences(valid_smiles[:500], beam_width=5)

# Compare predictions
count = 0
for s, p in zip(valid_smiles[:500], predicted_oh):
    if ''.join(p) == s:  # Join predicted tokens into a SMILES string
        count += 1
print(f'SeqToSeq_OneHot: Reproduced {count} of 500 validation SMILES strings')

SeqToSeq_OneHot: Reproduced 0 of 500 validation SMILES strings


Now we'll trying using the encoder as a way to generate molecular fingerprints.  We compute the embedding vectors for all molecules in the training and validation datasets, and create new datasets that have those as their feature vectors.  The amount of data is small enough that we can just store everything in memory.

In [ ]:
# Start timing
time_start = time.time()

train_embeddings_ind = model_ind.predict_embeddings(train_smiles)
train_embeddings_dataset_ind = dc.data.NumpyDataset(
    train_embeddings_ind,
    train_dataset.y,
    train_dataset.w.astype(np.float32),
    train_dataset.ids
)

valid_embeddings_ind = model_ind.predict_embeddings(valid_smiles)
valid_embeddings_dataset_ind = dc.data.NumpyDataset(
    valid_embeddings_ind,
    valid_dataset.y,
    valid_dataset.w.astype(np.float32),
    valid_dataset.ids
)

# End timing
time_end = time.time()

# Print the elapsed time
print(f"SeqToSeqModel (indices): Time taken for embedding computation and dataset creation: {time_end - time_start:.2f} seconds")

AttributeError: 'SeqToSeqModel' object has no attribute 'predict_embeddings'

In [ ]:
# Start timing
time_start = time.time()

# Generate embeddings for training and validation datasets
train_embeddings_oh = model_oh.predict_embeddings(train_smiles)
train_embeddings_dataset_oh = dc.data.NumpyDataset(
    train_embeddings_oh,
    train_dataset.y,
    train_dataset.w.astype(np.float32),
    train_dataset.ids
)

valid_embeddings_oh = model_oh.predict_embeddings(valid_smiles)
valid_embeddings_dataset_oh = dc.data.NumpyDataset(
    valid_embeddings_oh,
    valid_dataset.y,
    valid_dataset.w.astype(np.float32),
    valid_dataset.ids
)

# End timing
time_end = time.time()

# Print the elapsed time
print(f"SeqToSeq_OneHot: Time taken for embedding computation and dataset creation: {time_end - time_start:.2f} seconds")

Time taken for embedding computation and dataset creation: 3.36 seconds


For classification, we'll use a simple fully connected network with one hidden layer.

In [ ]:
from deepchem.models.fcnet import MultitaskClassifier

classifier_ind = MultitaskClassifier(
    n_tasks=len(tasks),
    n_features=256,
    layer_sizes=[512]
)

# Train the classifier
print(f"SeqToSeqModel: Average Loss")
classifier_ind.fit(train_embeddings_dataset_oh, nb_epoch=10)

0.2582392120361328

In [ ]:
classifier_oh = MultitaskClassifier(
    n_tasks=len(tasks),
    n_features=256,
    layer_sizes=[512]
)

# Train the classifier
print(f"SeqToSeq_OneHot: Average Loss")
classifier_oh.fit(train_embeddings_dataset_oh, nb_epoch=10)

Find out how well it worked.  Compute the ROC AUC for the training and validation datasets.

In [ ]:
metric = dc.metrics.Metric(dc.metrics.roc_auc_score, np.mean, mode="classification")

# SeqToSeqModel (indices)
train_score_ind = classifier_ind.evaluate(train_embeddings_dataset_ind, [metric], transformers)
valid_score_ind = classifier_ind.evaluate(valid_embeddings_dataset_ind, [metric], transformers)

print(f"SeqToSeqModel")
print(f"--------------------------------------")
print(f"Training set ROC AUC: {train_score_ind}")
print(f"Validation set ROC AUC: {valid_score_ind}")

# SeqToSeq_OneHot
train_score_oh = classifier_oh.evaluate(train_embeddings_dataset_oh, [metric], transformers)
valid_score_oh = classifier_oh.evaluate(valid_embeddings_dataset_oh, [metric], transformers)

print(f"SeqToSeq_OneHot")
print(f"--------------------------------------")
print(f"Training set ROC AUC: {train_score_oh}")
print(f"Validation set ROC AUC: {valid_score_oh}")

Training set ROC AUC: {'mean-roc_auc_score': 0.6681531784852774}
Validation set ROC AUC: {'mean-roc_auc_score': 0.5942762957849965}


# Congratulations! Time to join the Community!

Congratulations on completing this tutorial notebook! If you enjoyed working through the tutorial, and want to continue working with DeepChem, we encourage you to finish the rest of the tutorials in this series. You can also help the DeepChem community in the following ways:

## Star DeepChem on [GitHub](https://github.com/deepchem/deepchem)
This helps build awareness of the DeepChem project and the tools for open source drug discovery that we're trying to build.

## Join the DeepChem Gitter
The DeepChem [Gitter](https://gitter.im/deepchem/Lobby) hosts a number of scientists, developers, and enthusiasts interested in deep learning for the life sciences. Join the conversation!